# 05 · LLM context design

**Deck section 5** · slides 51–60

The context window is the boundary between retrieval and generation, and it is the one budget
with a hard wall. Everything in this section is an allocation decision: how many tokens each
part of the prompt gets, how many chunks survive, in what order, carrying what provenance.

**By the end you can**

- write a token budget with named slices, hard caps, and a stated overflow behaviour for each
- explain what every annotation in a packed evidence block is *for* at 2am
- size `k` against a token cap and a recall target instead of picking a round number
- measure your own system's position sensitivity rather than assuming the U-curve
- say when a RAG system should abstain, and what it costs when it cannot


In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / "raglab" / "__init__.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from raglab.bootstrap import bootstrap
bootstrap(verbose=False)

import numpy as np
import pandas as pd
import raglab
from raglab import (viz, tables, catalog, chunking, context, generate, metrics,
                     pipeline, retrieve)
viz.reset_figures("5."); tables.reset_tables("5.")

bundle, index, pipe = raglab.quickstart(**raglab.TUNED)
dev = [q for q in bundle.questions if q.slice == "dev"]
sweep = dev[:90]
answerable = [q for q in sweep if q.question_type != "null"]
print(f"{len(sweep)} questions in the sweep set")

---

## 5.1 Allocate the window before you fill it

A 32k working context is not 32k of evidence. It is six named slices, each with a cap and
each with a different failure when it overflows — and only one of those failures is visible
to the user.


In [ ]:
viz.budget([(s.name.split("+")[0].strip()[:22], s.cap) for s in context.DEFAULT_BUDGET],
           unit="tokens", title="Allocating a 32k working context",
           kicker="Engineering budget",
           caption="Write these into a config with hard caps. Without a cap, evidence expands "
                   "until something else is silently truncated.",
           source="Deck slide 53")

tables.show(pd.DataFrame([[s.name, f"{s.cap:,}", s.on_overflow] for s in context.DEFAULT_BUDGET],
                         columns=["Slice", "Cap (tokens)", "What happens when it overflows"]),
            title="Six slices, six different failures",
            kicker="Budget contract",
            caption="Only the output-reserve row produces a failure the user can see. The "
                    "other five degrade quietly, which is why each needs a cap rather than "
                    "an intention.",
            emphasize="Slice")

In [ ]:
q = next(x for x in answerable if x.hops >= 2)
trace = pipe.run(q.query, qid=q.qid)
packed = trace._packed_obj

used = packed.tokens
caps = {s.name: s.cap for s in context.DEFAULT_BUDGET}
rows = [
    ["System instructions + output contract", used["system"], caps["System instructions + output contract"]],
    ["Tool / schema definitions", used["tools"], caps["Tool / schema definitions"]],
    ["User query + conversation state", used["question"] + used["conversation"],
     caps["User query + conversation state"]],
    ["Retrieved evidence, k chunks", used["evidence"], caps["Retrieved evidence, k chunks"]],
]
frame = pd.DataFrame(rows, columns=["Slice", "Used on this query", "Cap"])
frame["Headroom"] = frame["Cap"] - frame["Used on this query"]
frame["Utilisation"] = (frame["Used on this query"] / frame["Cap"]).map(lambda v: f"{v:.0%}")
tables.show(frame, title=f"One real query against the budget",
            kicker="Measured", emphasize="Utilisation",
            caption=f"k={pipe.cfg.k} under a {pipe.cfg.evidence_token_cap:,}-token evidence "
                    "cap. The cap binds long before the model's context window does — which "
                    "is the point: you are not managing the window, you are managing cost.")

---

## 5.2 What a packed context actually looks like

Read the annotations right to left. Every one of them is a debugging affordance you will want
at 2am, and every one costs a handful of tokens.


In [ ]:
block = packed.blocks[0]
viz.annotated_text([
    {"text": context.SYSTEM_CONTRACT, "color": "#F0C674",
     "note_title": "Stable prefix first",
     "note": "Instructions and the output contract never change per query, so they sit at "
             "the front and stay cacheable. Notebook 07 measures what that is worth."},
    {"text": "\nQUESTION\n" + trace.query, "color": "#8ABEB7",
     "note_title": "The question, verbatim",
     "note": "Included unchanged, or explicitly rewritten. Never silently dropped when the "
             "conversation slice overflows."},
    {"text": "\nEVIDENCE\n" + block["rendered"][:520], "color": "#D8DEE6",
     "note_title": "[S#] not a URL",
     "note": "Short opaque IDs are cheap to emit and easy to resolve. Long URLs invite the "
             "model to paraphrase them wrongly. doc_id + chunk ordinal is what lets you "
             "replay the exact retrieval and diff two runs. The date is in the block because "
             "a temporal question is unanswerable if publication dates only live in the "
             "index."},
    {"text": "\n" + context.OUTPUT_CONTRACT, "color": "#B294BB",
     "note_title": "One exact abstention token",
     "note": "A fixed string is parseable. “I'm not sure” is not — and every downstream "
             "metric that counts refusals depends on being able to parse it."},
], title="What a packed context actually looks like",
   kicker="Worked example",
   caption="Six annotations, a few dozen tokens, and the difference between a system you can "
           "debug and one you can only apologise for.",
   source="Deck slide 55")

In [ ]:
# Provenance is only useful if it resolves. Check that it does, over the whole set.
resolvable, on_gold, uncited = 0, 0, 0
checked = 0
for x in answerable:
    tr = pipe.run(x.query, qid=x.qid, acl_groups=bundle.personas.get(x.persona))
    gm, _ = metrics.resolve_gold(x, pipe.chunks)
    gold = {c for s in gm.values() for c in s}
    cites = [tr._packed_obj.resolve(c) for c in tr.citations]
    cites = [c for c in cites if c]
    if not tr.citations:
        uncited += 1
        continue
    checked += 1
    resolvable += len(cites) / len(tr.citations)
    on_gold += sum(1 for c in cites if c in gold) / max(1, len(cites))

print(f"answers with at least one citation   {checked}/{len(answerable)}")
print(f"citations that resolve to a packed chunk   {resolvable/max(1,checked):.3f}")
print(f"citations that land on gold evidence       {on_gold/max(1,checked):.3f}")
print(f"answers with no citation at all            {uncited}")
print("\nThose are two different failures. An unresolvable ID is a bug. A resolvable ID on a")
print("chunk that does not support the claim is a trust problem — and it is the one clients")
print("notice, because a wrong citation is worse than no citation.")

---

## 5.3 Sizing k against a token cap

`N` and `k` are different knobs with different owners: retrieval owns `N`, the prompt owns
`k`. A small `k` omits a necessary evidence hop; a large `k` buys tokens, distractors and
position risk. The only way to pick one is to sweep it against the metric that matters and
the budget that binds.


In [ ]:
viz.flow([("Top-N candidates", "retrieval owns this"), ("Rerank", "→ ordered"),
          ("Deduplicate and filter", "duplicates are distractors that also cost tokens"),
          ("Pack top-k evidence", "the prompt owns this"),
          ("LLM context window", "hard wall")],
         title="Top-k: optimise evidence coverage under a token budget",
         kicker="Section 5 · packing", source="Deck slide 52",
         caption="Tune N, k and the evidence-token cap together; evaluate recall and answer "
                 "quality separately.")

In [ ]:
ks = [2, 3, 5, 8, 12, 16, 24]
er, fcr, cp, toks, correct = [], [], [], [], []
for k in ks:
    rs = pipeline.evaluate(pipe.variant(f"k={k}", k=k), sweep, pipe.chunks,
                           personas=bundle.personas)
    s = metrics.summarize(rs)
    er.append(s["evidence_recall"]); fcr.append(s["full_chain_recall"])
    cp.append(s["context_precision"]); correct.append(s["answer_correct"])
    toks.append(s["tokens_in"])

viz.lines(ks, {"evidence recall": er, "full-chain recall": fcr,
               "context precision": cp, "answer correctness": correct},
          title="Sweeping k: what you buy and what you dilute",
          kicker="Measured", xlabel="k (chunks packed)", ylabel="score",
          caption="Recall rises and keeps rising. Context precision falls monotonically — "
                  "every extra slot is more likely to hold a distractor than a gold chunk.")

In [ ]:
cap = pipe.cfg.evidence_token_cap
frame = pd.DataFrame({
    "k": ks,
    "evidence recall": [round(v, 3) for v in er],
    "full-chain": [round(v, 3) for v in fcr],
    "context precision": [round(v, 3) for v in cp],
    "prompt tokens (avg)": [round(t) for t in toks],
    "share of 6k evidence cap": [f"{(t - toks[0]) / cap:.0%}" for t in toks],
})
frame["marginal full-chain per +1k tokens"] = [""] + [
    f"{(fcr[i] - fcr[i-1]) / max(1e-6, (toks[i] - toks[i-1]) / 1000):+.3f}"
    for i in range(1, len(ks))]
tables.show(frame, title="k as a purchase, priced per thousand tokens",
            kicker="Sizing decision",
            caption="The last column is the one to argue from. It falls off a cliff, and the "
                    "point where it does is your operating point — not a round number "
                    "somebody liked.",
            emphasize="marginal full-chain per +1k tokens")

In [ ]:
viz.scatter_frontier([(f"k={k}", toks[i], fcr[i]) for i, k in enumerate(ks)],
                     xlabel="average prompt tokens", ylabel="full-chain recall",
                     title="The cost/quality frontier, with the operating point marked",
                     kicker="Frontier", chosen=f"k={pipe.cfg.k}",
                     caption="'A stated cost/quality frontier with the chosen operating point "
                             "marked' is what the build rubric asks for to exceed the bar on "
                             "cost and latency. This is that artefact.")

---

## 5.4 Position inside the context

Liu et al. (2023) reported a U-shape: holding the evidence set constant and moving only the
position of the relevant document, accuracy is strongest when the gold document sits at the
start or the end, and weakest in the middle.

### Being careful here

Our offline reader is extractive and has no attention mechanism, so it has **no position
sensitivity of its own** — and that is worth demonstrating rather than hiding, because it
makes the point precisely. What the code below measures is *your* system's position
sensitivity, using the deck's prescribed method: force the gold chunk into position 1, the
middle, and last, and compare. On this reader the answer is "none". Point the same harness at
Bedrock and the number will not be zero, and the gap is what you would report.


In [ ]:
def place_gold_at(hits, gold_ids, position):
    '''Force a gold chunk into a chosen slot, keeping the evidence set identical.'''
    gold_hits = [h for h in hits if h.chunk_id in gold_ids]
    rest = [h for h in hits if h.chunk_id not in gold_ids]
    if not gold_hits:
        return None
    g = gold_hits[0]
    others = (rest + gold_hits[1:])[: pipe.cfg.k - 1]
    if position == "first":
        return [g] + others
    if position == "last":
        return others + [g]
    mid = len(others) // 2
    return others[:mid] + [g] + others[mid:]


positions = ("first", "middle", "last")
scores = {p: [] for p in positions}
tested = 0
for x in answerable[:45]:
    gm, _ = metrics.resolve_gold(x, pipe.chunks)
    gold = {c for s in gm.values() for c in s}
    if not gold:
        continue
    cands = pipe.retriever.search(x.query, pipe.cfg)
    ranked = pipe.reranker.rerank(x.query, cands, depth=pipe.cfg.rerank_depth)
    if not any(h.chunk_id in gold for h in ranked[:50]):
        continue
    tested += 1
    for p in positions:
        arranged = place_gold_at(ranked[:50], gold, p)
        pk = context.build_prompt(x.query, arranged, k=pipe.cfg.k,
                                  token_cap=pipe.cfg.evidence_token_cap)
        ans = pipe.generator.generate(x.query, pk)
        scores[p].append(metrics.answer_correct(ans.text, x.answer))

viz.bars(list(positions), {"answer correctness": [sum(scores[p]) / len(scores[p])
                                                  for p in positions]},
         title=f"Position sensitivity of the offline reader ({tested} questions)",
         kicker="Measured, not assumed", ylabel="answer correctness",
         caption="Identical evidence set, only the gold chunk's slot changes. A flat result "
                 "here is the correct result for an extractive reader — and the same harness "
                 "against a real model is how you would find your own number.")

spread = max(sum(scores[p]) / len(scores[p]) for p in positions) - \
         min(sum(scores[p]) / len(scores[p]) for p in positions)
print(f"position sensitivity (max − min) = {spread:.3f}")

In [ ]:
tables.show(pd.DataFrame([
    ["Keep k small", "Fewer chunks in the middle at all",
     f"Measured in §5.3: k={pipe.cfg.k} versus k=24 changes context precision from "
     f"{cp[ks.index(pipe.cfg.k)]:.3f} to {cp[-1]:.3f}", "Free — it also cuts cost"],
    ["Order by reranker score, then interleave",
     "Place the two highest-scoring chunks at the head and the tail of the evidence block",
     "`retrieve.order_for_position(hits, 'edges')` — one line in the packer",
     "Free"],
    ["Restate the question after the evidence",
     "A short recap at the tail puts the task in the strong position",
     "Already in `context.build_prompt(restate_question=True)`",
     "~15 tokens"],
    ["Test it, do not assume it",
     "Force the gold chunk into position 1, mid and last on your own eval set",
     f"Done above: spread = {spread:.3f} on this reader",
     "One sweep, and it tells you whether the other three are worth doing at all"],
], columns=["Mitigation", "What it does", "Where it is in this toolkit", "What it costs"]),
    title="What to do about the position effect",
    kicker="Mitigations", source="Deck slide 57", emphasize="Mitigation",
    highlight_rows=lambda r: r["Mitigation"].startswith("Test it"))

In [ ]:
edge = pipeline.evaluate(pipe.variant("edges", order="edges"), sweep, pipe.chunks,
                         personas=bundle.personas)
score_order = pipeline.evaluate(pipe.variant("score", order="score"), sweep, pipe.chunks,
                                personas=bundle.personas)
tables.show(pipeline.compare_runs({"order by score": score_order, "edge interleave": edge},
                                  keys=("evidence_recall", "full_chain_recall",
                                        "answer_correct")),
            title="Edge interleaving on a reader with no position sensitivity",
            kicker="Control", emphasize="run",
            caption="Identical, as it must be — the packer reordered chunks a reader that "
                    "does not care about order then read. Keeping the control in the notebook "
                    "is how you avoid attributing a real gain to the wrong cause later.")

---

## 5.5 "Why not just put everything in the context window?"

A one-million-token window does not delete retrieval. It changes where the tradeoff sits.


In [ ]:
catalog.LONG_CONTEXT.show()

In [ ]:
from raglab import costs

corpus_tokens = sum(chunking.approx_tokens(d.body) for d in bundle.documents)
rates = costs.Rates()
stuff = rates.cost(input_tokens=corpus_tokens, output_tokens=450)
retrieve_cost = costs.unit_economics(k=pipe.cfg.k)["total_per_query"]

tables.show(pd.DataFrame([
    ["Stuff the window", f"{corpus_tokens:,}", f"${stuff:.4f}",
     f"${stuff * 200_000:,.0f}", "scales with corpus size — a per-query bill, not a one-off"],
    ["Retrieve, then generate", f"{int(pipe.cfg.k * 550 + 3200):,}", f"${retrieve_cost:.4f}",
     f"${retrieve_cost * 200_000:,.0f}", "roughly flat in corpus size; index cost paid once"],
], columns=["Approach", "Input tokens/query", "Cost/query", "At 200k queries/month",
            "How it scales"]),
    title=f"The same corpus, both ways",
    kicker="Priced",
    caption=f"This corpus is only {corpus_tokens:,} tokens — small enough that stuffing is "
            "genuinely viable, which is exactly when you should do it. Multiply the corpus by "
            "fifty and the first row multiplies with it while the second does not.",
    emphasize="Cost/query")

tables.callout(
    "The honest answer in a client conversation: <b>stuff the window for a small, "
    "slow-changing, single-tenant corpus and ship this week.</b> Move to retrieval when cost "
    "per query, attribution, or access control becomes the binding constraint — and expect "
    "that to happen sooner than the client thinks. Note which of those three is usually first: "
    "it is access control, and it arrives the moment a second user with different permissions "
    "appears.", kind="note", title="What to actually say")

---

## 5.6 Generation controls, and the abstention problem

Four controls, evaluated separately from retrieval — because correct evidence plus a bad
contract still produces a bad answer.


In [ ]:
tables.show(pd.DataFrame([
    ["Evidence constraint", "Answer only from the supplied context; do not fill gaps from "
     "unsupported assumptions", "faithfulness / groundedness", "notebook 06"],
    ["Abstention policy", "State that evidence is insufficient when required facts are absent "
     "or conflict", "abstention precision/recall on the null set", "§5.7 below"],
    ["Citation requirement", "Attach source IDs to factual claims where the product needs "
     "traceability", "citation accuracy — resolvable, and on-gold", "§5.2 above"],
    ["Output contract", "Enforce a schema when downstream systems consume the answer",
     "schema validity rate", "notebook 09"],
], columns=["Control", "What it says", "How you measure it", "Where it is measured here"]),
    title="Generation controls for grounded answers",
    kicker="Section 5 · generation", source="Deck slide 56", emphasize="Control")

In [ ]:
rows = pipeline.evaluate(pipe, bundle.questions, pipe.chunks, personas=bundle.personas)
ab = metrics.abstention_scores(rows)
nulls = [r for r in rows if r["is_null"]]
print(f"null questions            {len(nulls)}")
print(f"abstained                 {sum(1 for r in nulls if r['abstained'])}")
print(f"abstention recall         {ab['abstention_recall']:.3f}")
print(f"answered anyway           {ab['false_answers_on_null']}")
print("\nThe contract is in the prompt. The model reads it. And with no threshold behind it,")
print("nothing enforces it — the reader finds a plausible sentence and quotes it.")

### Can a score threshold fix it?

The obvious move is a retrieval-score cut-off: if the best evidence scores below θ, abstain.
`ExtractiveGenerator(min_evidence_score=θ)` implements exactly that. Sweep θ and read the
precision/recall curve — which the build rubric asks for by name.


In [ ]:
# Abstention by threshold is a post-hoc decision on one number per question, so there is no
# reason to re-run the pipeline once per candidate θ. Run it once on the FULL eval set —
# the base rate matters, and a stratified subsample would flatter the precision — then sweep
# θ in numpy.
tops, is_null = [], []
for x in bundle.questions:
    t = pipe.run(x.query, qid=x.qid, acl_groups=bundle.personas.get(x.persona))
    tops.append(max((b["score"] for b in t.packed), default=0.0))
    is_null.append(x.question_type == "null")
tops, is_null = np.array(tops), np.array(is_null)
print(f"{len(tops)} questions · {is_null.sum()} null ({is_null.mean():.0%} base rate)")

thetas = np.round(np.arange(0.30, 0.96, 0.02), 2)
prec, rec, f1 = [], [], []
for th in thetas:
    abstain = tops < th                       # below θ → refuse to answer
    tp = int((abstain & is_null).sum())
    fp = int((abstain & ~is_null).sum())
    fn = int((~abstain & is_null).sum())
    p_ = tp / (tp + fp) if (tp + fp) else 0.0
    r_ = tp / (tp + fn) if (tp + fn) else 0.0
    prec.append(p_); rec.append(r_)
    f1.append(2 * p_ * r_ / (p_ + r_) if (p_ + r_) else 0.0)

best_i = int(np.argmax(f1))
viz.lines(list(thetas), {"abstention precision": prec, "abstention recall": rec, "F1": f1},
          title="Abstention threshold: the precision/recall curve",
          kicker="Measured · full eval set, real base rate",
          xlabel="θ — minimum top evidence score required to answer", ylabel="score",
          vline=float(thetas[best_i]), vline_label=f"best F1 = {f1[best_i]:.2f}",
          caption="A threshold that separates the two populations would show precision "
                  "holding while recall rises. Read what this one does instead — and note "
                  "that the base rate is doing a lot of the work at the right-hand end.")
print(f"best F1 {f1[best_i]:.3f} at θ={thetas[best_i]}  "
      f"(precision {prec[best_i]:.3f}, recall {rec[best_i]:.3f})")
print(f"at that θ the system refuses {int((tops < thetas[best_i]).sum())} of {len(tops)} "
      f"questions — {int(((tops < thetas[best_i]) & ~is_null).sum())} of them answerable")

In [ ]:
# Why it fails -- look at the two distributions rather than guessing.
null_top = tops[is_null]
ans_top = tops[~is_null]

viz.bars(["null (unanswerable)", "answerable"],
         {"median top evidence score": [float(np.median(null_top)), float(np.median(ans_top))],
          "25th percentile": [float(np.percentile(null_top, 25)),
                              float(np.percentile(ans_top, 25))],
          "75th percentile": [float(np.percentile(null_top, 75)),
                              float(np.percentile(ans_top, 75))]},
         title="The two distributions a threshold would have to separate",
         kicker="Diagnosis", ylabel="top packed evidence score",
         caption="They overlap almost completely. No horizontal line through this chart "
                 "separates the two populations, so no threshold can.")

In [ ]:
overlap = (float(np.percentile(null_top, 25)) < float(np.percentile(ans_top, 75))
           and float(np.percentile(ans_top, 25)) < float(np.percentile(null_top, 75)))
tables.callout(
    f"<b>The best F1 this threshold can reach is {f1[best_i]:.2f}, at a cost of "
    f"{int(((tops < thetas[best_i]) & ~is_null).sum())} answerable questions refused.</b> "
    "That is not a usable abstention policy, and the distributions above show why: they "
    f"overlap across the interquartile range{' ' if overlap else ' only partly '}"
    "— so any line you draw trades one failure directly for the other. "
    "We tried the reranker score, and in building this curriculum also corpus-IDF coverage, "
    "sentence-level rare-term coverage, and a conjunctive corpus-presence check. All four sit "
    "near chance."
    "<br><br>We also tried corpus-IDF coverage, sentence-level rare-term coverage and a "
    "conjunctive corpus-presence check while building this curriculum. All of them sit near "
    "chance, and the reason is visible once you look at the questions rather than the scores. The "
    "null questions name <i>real entities in the corpus's own vocabulary</i> — “Which "
    "organization acquired Halcyon Robotics?” — while the answerable ones paraphrase and use "
    "descriptors. So the unanswerable questions are, lexically and semantically, <i>closer</i> "
    "to the corpus than the answerable ones. A similarity threshold is measuring the wrong "
    "thing entirely."
    "<br><br><b>Abstention is an entailment question, and entailment needs a reader.</b> That "
    "is why the deck puts abstention in <i>generation controls</i> and the null set in "
    "<i>evaluation</i>, and never in the retriever. The fixes that work are a prompt contract "
    "with one exact refusal token, a cheap sufficiency check as a separate call (notebook 08), "
    "and a judge that verifies the refusal behaviour held (notebook 06) — plus, in a regulated "
    "product, an inline guardrail that blocks before the reply is sent.",
    kind="warn", title="A negative result worth more than a positive one")

---

## 5.7 Failure points and the interview


In [ ]:
k2 = pipeline.evaluate(pipe.variant("k=2", k=2), sweep, pipe.chunks, personas=bundle.personas)
k16 = pipeline.evaluate(pipe.variant("k=16", k=16), sweep, pipe.chunks,
                        personas=bundle.personas)
base = metrics.summarize(pipeline.evaluate(pipe, sweep, pipe.chunks,
                                           personas=bundle.personas))

tables.show(pd.DataFrame([
    ["Top-k too small", f"full-chain {metrics.summarize(k2)['full_chain_recall']:.3f} at k=2 "
     f"versus {base['full_chain_recall']:.3f} at k={pipe.cfg.k}",
     "A two-document answer receives only the first evidence hop. Multi-hop questions "
     "collapse first and hardest."],
    ["Context overload", f"context precision "
     f"{metrics.summarize(k16)['context_precision']:.3f} at k=16 versus "
     f"{base['context_precision']:.3f} at k={pipe.cfg.k}",
     "Low-value chunks bury the highest-ranked source and cost tokens doing it."],
    ["Provenance loss", f"citations that resolve: {resolvable/max(1,checked):.3f}; "
     f"citations landing on gold: {on_gold/max(1,checked):.3f}",
     "Concatenated snippets make a citation point at the wrong document. A wrong citation is "
     "worse than no citation."],
    ["No abstention", f"abstention recall {ab['abstention_recall']:.3f}, "
     f"{ab['false_answers_on_null']} null questions answered anyway",
     "The model fills an evidence gap with a plausible but unsupported claim — and no other "
     "metric on the report moves."],
], columns=["Signature", "Measured here", "What it costs"]),
    title="Failure points: LLM context design", kicker="Failure points",
    source="Deck slide 59", emphasize="Signature")

In [ ]:
tables.show(pd.DataFrame([
    [catalog.SECTION_QUESTIONS[5][0],
     "Whether you ask what the SLA is before answering",
     "Ask for the latency and cost envelope first, then show the sweep: marginal full-chain "
     "recall per thousand tokens, with the operating point marked on a frontier. §5.3 is that "
     "artefact."],
    [catalog.SECTION_QUESTIONS[5][1],
     "Whether you have debugged a citation at 2am",
     "A short opaque ID, the doc_id and chunk ordinal, the title, the publication date, the "
     "score, and a delimiter that preserves the document boundary. The date because temporal "
     "questions are otherwise unanswerable; the ordinal because it is what makes two runs "
     "diffable."],
    [catalog.SECTION_QUESTIONS[5][2],
     "Whether you would measure it or cite the paper",
     "Force the gold chunk into position 1, mid and last on your own eval set and report the "
     "spread. Then mitigate in order of cost: smaller k first, edge interleaving second, "
     "restating the question third."],
    [catalog.SECTION_QUESTIONS[5][3],
     "Whether you know a threshold is not enough",
     "When the evidence does not entail an answer — which is an entailment judgment, not a "
     "similarity score. Show the PR curve, show the overlapping distributions, then propose "
     "the contract plus a sufficiency check plus a judged null set. §5.6."],
], columns=["Question", "What the panel is testing", "What a strong answer covers"]),
    title="Typical interview questions: LLM context design",
    kicker="Section 5 · interview", source="Deck slide 60", emphasize="Question")

---

## What carries forward

- The budget is six named slices with hard caps and a stated overflow behaviour each. Five of
  the six fail silently.
- Every annotation in an evidence block earns its tokens by being something you need while
  debugging. `doc_id + ordinal` is what makes two runs diffable.
- `k` is a purchase. Price it per thousand tokens and mark the operating point on a frontier.
- Measure your own position sensitivity before mitigating it — and keep the control.
- Abstention cannot be bought with a similarity threshold. It needs a contract, a reader, and
  a null set that scores it.

**Next:** `06_evaluation_approaches.ipynb` — layered metrics, a judge you calibrate with
Cohen's κ and attack with bias probes, and the release gate that decides whether any of this
ships.
